# Step 15.3 — MiniGPT Top-Level Architecture：Decoder-Only GPT 最外层到底由哪些部分组成？

Step 15.2 已经解释为什么 autoregressive language modeling 可以只保留一条 causal token stream。

现在把我们已经做好的 Transformer Block 放到真正的 MiniGPT 顶层结构里：

```text
token IDs
   ↓
token embedding
   +
position embedding
   ↓
initial hidden states
   ↓
Block × N
   ↓
final norm
   ↓
LM head
   ↓
vocabulary logits
```

Shape 主线：

$$
\boxed{
(B,T)
\rightarrow
(B,T,C)
\rightarrow
\cdots
\rightarrow
(B,T,C)
\rightarrow
(B,T,V)
}.
$$

本节只解释 top-level architecture 与各组件职责，并准备组件级 shape code。

不实现 `MiniGPT(nn.Module)`，不进入 Step 15.4。


## 1. 顶层新记号：V、C、T、T_max、N_blocks

我们现在需要把之前分散出现的 shape symbols 放在一起：

```text
B        = batch size
T        = 当前输入 sequence length
V        = vocabulary size
C        = model / embedding width
T_max    = maximum context positions supported by learned position table
N_blocks = Transformer Block 数量
```

输入首先仍然是 integer token IDs：

$$
\boxed{
tokens
\in
\mathbb Z^{B\times T}
}.
$$

Token ID 只是 categorical index，本身不是连续语义 feature。

真正进入 Transformer Blocks 之前，需要先把每个 ID 转成 C-dimensional representation。


## 2. Token Embedding：`token ID -> C-dimensional vector`

定义 learned token embedding table：

$$
E_{tok}
\in
\mathbb R^{V\times C}.
$$

对于：

$$
tokens[b,t]=k,
$$

lookup：

$$
\boxed{
X_{tok}[b,t,:]
=
E_{tok}[k,:]
}.
$$

所以：

$$
(B,T)
\xrightarrow{TokenEmbedding}
\boxed{(B,T,C)}.
$$

MLX 0.32.2：

```python
nn.Embedding(
    num_embeddings=V,
    dims=C,
)
```

stored weight shape：

$$
(V,C).
$$

这是 lookup table，不需要显式构造 V-dimensional one-hot vectors。


## 3. Position Embedding：Token 是什么，还不够；还要知道它在哪里

Self-Attention 本身主要根据 representation 内容建立 pairwise interaction。

Causal mask 告诉模型：

> 哪些 positions **允许被看见**。

但 causal visibility rule 不等于完整的 learned positional representation。

本项目的 MiniGPT 顶层采用最简单的 **learned absolute position embedding**：

$$
E_{pos}
\in
\mathbb R^{T_{max}\times C}.
$$

当前位置 IDs：

$$
0,1,\ldots,T-1.
$$

lookup 后：

$$
X_{pos}
\in
\mathbb R^{T\times C}.
$$

要求：

$$
\boxed{T\le T_{max}}.
$$

这是一种教学/架构选择，不代表所有现代 LLM 都使用 learned absolute position embeddings。


## 4. 为什么 Token Embedding 和 Position Embedding 是相加？

Token embedding：

$$
X_{tok}:(B,T,C).
$$

Position embedding：

$$
X_{pos}:(T,C).
$$

`(T,C)` 可以沿 batch dimension broadcast，于是：

$$
\boxed{
X_0
=
X_{tok}
+
X_{pos}
}
$$

得到：

$$
X_0:(B,T,C).
$$

直觉上，每个 position 的初始 representation 同时包含：

```text
what token is this?
+
where is this token?
```

注意 addition 会把两类信息叠加到同一个 C-dimensional space 中；它不是 lossless concatenation，也不保证之后能从和中唯一分离两部分。


## 5. Tiny Example：Token 信息 + Position 信息

假设一个 sample 有三个 positions，`C=2`。

Token embeddings：

$$
X_{tok}
=
\begin{bmatrix}
1&0\\
0&1\\
1&1
\end{bmatrix}.
$$

Position embeddings：

$$
X_{pos}
=
\begin{bmatrix}
0.1&0.0\\
0.0&0.2\\
0.3&0.3
\end{bmatrix}.
$$

相加：

$$
X_0
=
\begin{bmatrix}
1.1&0.0\\
0.0&1.2\\
1.3&1.3
\end{bmatrix}.
$$

同一个 token 如果出现在不同 position，会加上不同 position vector，因此进入 Block 的初始 representation 可以不同。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

B = 2
T = 4
V = 10
C = 6
T_max = 8

token_ids = mx.array([
    [0, 1, 2, 3],
    [3, 2, 1, 0],
])  # (B,T), integer categorical IDs

token_embedding = nn.Embedding(
    num_embeddings=V,
    dims=C,
)
position_embedding = nn.Embedding(
    num_embeddings=T_max,
    dims=C,
)

assert T <= T_max

position_ids = mx.arange(T)  # (T,)

token_features = token_embedding(token_ids)       # (B,T,C)
position_features = position_embedding(position_ids)  # (T,C)
hidden = token_features + position_features       # broadcast -> (B,T,C)

print('token IDs:', token_ids.shape)
print('token embedding weight:', token_embedding.weight.shape)
print('position embedding weight:', position_embedding.weight.shape)
print('token features:', token_features.shape)
print('position features:', position_features.shape)
print('initial hidden:', hidden.shape)

# Expected shapes only; this master notebook remains unexecuted.


## 6. `Block × N`：为什么一个 Block 之后还要继续堆？

Step 14.5 的一个 Block：

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

因此它可以反复堆叠：

$$
X_0
\xrightarrow{Block_0}
X_1
\xrightarrow{Block_1}
\cdots
\xrightarrow{Block_{N-1}}
X_N.
$$

每层都保持：

$$
X_l\in\mathbb R^{B\times T\times C}.
$$

通常 `Block × N` 指 **N 个结构相同、但拥有各自 learned parameters 的 Block instances**，不是默认反复调用同一个共享权重 Block。

随着层数增加，representation 会经过多轮：

```text
causal cross-position mixing
+
within-position nonlinear transformation
```

但本节只解释 stack 角色，不在这里实现 block list；那属于 Step 15.4。


## 7. Final Norm：为什么 Blocks 后面还要再 Normalize 一次？

我们 Step 14.5 使用 Pre-Norm Block：

$$
x_1=x+Attn(N_1(x)),
$$

$$
y=x_1+MLP(N_2(x_1)).
$$

注意 Block 最终返回的是 residual stream `y`；第二次 residual add 之后没有再自动做一个 output normalization。

因此本项目 MiniGPT 顶层在所有 Blocks 后放：

$$
\boxed{
X_{final}
=
N_{final}(X_N)
}.
$$

我们延续当前教学选择使用：

```python
nn.RMSNorm(dims=C)
```

Shape 保持：

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

Final norm 是本项目 Pre-Norm design 的组成部分；不要把它理解成所有 Transformer architecture 都必须以完全相同方式放置 normalization。


## 8. LM Head：把每个 Position 的 C-dimensional State 变成 Vocabulary Logits

Transformer Blocks 输出的每个 position 仍是：

$$
h_{b,t}\in\mathbb R^C.
$$

但 next-token prediction 最终必须对 vocabulary 中 V 个 token 给出分数。

所以使用 **Language-Model Head / LM Head**：

$$
W_{LM}
\in
\mathbb R^{C\times V}.
$$

数学上：

$$
\boxed{
logits_{b,t}
=
h_{b,t}W_{LM}
}
$$

因此：

$$
(B,T,C)
\rightarrow
\boxed{(B,T,V)}.
$$

`logits[b,t,v]` 是 position `t` 对 vocabulary token `v` 的**未归一化分数**。

本节不在模型顶层手动加 softmax，也不进入 loss；next-token cross-entropy 属于后续训练章节。


In [ ]:
# COMPONENT-LEVEL ENDPOINT SHAPES ONLY — no Block stack is implemented here.

final_norm = nn.RMSNorm(dims=C)
lm_head = nn.Linear(
    input_dims=C,
    output_dims=V,
    bias=False,
)

# Placeholder representing the output of N already-existing Blocks.
hidden_after_blocks = mx.zeros((B, T, C))

normalized = final_norm(hidden_after_blocks)  # (B,T,C)
logits = lm_head(normalized)                  # (B,T,V)

print('hidden after Blocks:', hidden_after_blocks.shape)
print('final normalized:', normalized.shape)
print('LM-head stored weight:', lm_head.weight.shape)
print('logits:', logits.shape)

# Expected:
# final_norm keeps (B,T,C)
# lm_head.weight has shape (V,C)
# logits has shape (B,T,V)
#
# This is not a MiniGPT implementation and is not executed in the master.


## 9. `(B,T,V)` Logits 中每一行到底预测什么？

训练仍然使用 Step 15.2 的 shifted next-token setup。

例如：

```text
input positions:
<BOS>  I     love   coffee

labels:
I      love  coffee <EOS>
```

所以：

$$
logits[:,i,:]
\in
\mathbb R^V
$$

用于预测 input position `i` 后面的：

$$
x_{i+1}.
$$

因此虽然 output tensor 为所有 T 个 positions 同时产生 logits，causal mask 已经保证每个 position 的 hidden state 只依赖合法 prefix。

这就是：

```text
one causal forward
-> T sets of next-token logits
```

而不是只有最后一个 position 才能用于训练。


## 10. MiniGPT Top-Level Shape Chain

把全部顶层步骤串起来：

```text
token IDs
(B,T)

↓ token embedding

(B,T,C)

+ position embedding
(T,C) -> broadcast across B

↓

initial hidden
(B,T,C)

↓ Block × N

hidden after stack
(B,T,C)

↓ final RMSNorm

(B,T,C)

↓ LM head C→V

logits
(B,T,V)
```

这条链条里：

- `T` 一直保持不变；
- hidden width 在 Transformer stack 内一直保持 C；
- 只有最后 LM head 才把 feature axis 从 C 映射到 vocabulary axis V。


## 11. 五个顶层组件各自解决什么问题？

| 组件 | 输入 → 输出 | 核心职责 |
|---|---|---|
| Token Embedding | `(B,T) -> (B,T,C)` | 把 categorical token IDs 映射到 learned continuous representations |
| Position Embedding | `(T) -> (T,C)` | 提供 absolute position-specific learned signal |
| Block × N | `(B,T,C) -> (B,T,C)` | 反复进行 causal context mixing + nonlinear feature transformation |
| Final Norm | `(B,T,C) -> (B,T,C)` | 在 LM head 前规范最终 residual-stream feature scale |
| LM Head | `(B,T,C) -> (B,T,V)` | 为 vocabulary 中每个 next-token candidate 产生 logits |

这就是 roadmap 里所谓的 **MiniGPT top-level architecture**。

此时我们已经知道“有哪些模块、为什么存在、shape 怎样流动”，但还没有把它们封装成一个真正的 MLX `MiniGPT` class。


## 12. Position Embedding 的一个准确边界

本节为了建立最清晰的 MiniGPT，选择：

$$
\text{learned absolute position embedding}.
$$

因此需要一个最大位置表：

$$
E_{pos}:(T_{max},C).
$$

并要求当前：

$$
T\le T_{max}.
$$

但 positional information 有很多其他设计。

所以不要从本节推出：

> “所有 GPT / LLM 都必须拥有一个 `(T_max,C)` learned position table。”

本节只固定我们后续 MiniGPT 实现的简洁选择。


## 13. Step 15.3 材料边界

- MiniGPT 顶层 shape chain：`(B,T) -> token+position representations (B,T,C) -> Block×N -> final norm -> LM head -> (B,T,V)`。
- Token embedding table shape 为 `(V,C)`；token ID 是 categorical index，不是连续 feature。
- 本项目选择 learned absolute position embedding `(T_max,C)`；当前 position IDs `0...T-1` lookup 成 `(T,C)` 并 broadcast 加到 `(B,T,C)` token embeddings 上，要求 `T<=T_max`。
- causal mask 与 position embedding 职责不同：mask 管 visibility，position embedding 提供 position-specific representation signal。
- `Block × N` 通常指 N 个独立参数的 Block instances，每层保持 `(B,T,C)`；本节不实现 Block container。
- 对当前 Pre-Norm MiniGPT design，在 Block stack 后使用 final RMSNorm，shape 仍为 `(B,T,C)`。
- LM head 把 C 映射到 vocabulary width V，输出 `(B,T,V)` logits；本节不加 softmax、不讲 loss。
- 本轮只准备 Step 15.3；不进入 Step 15.4，不实现 `MiniGPT(nn.Module)`、不组装真实 Block list、不做 top-level shape assertions，也不处理 causal mask state-registration 等实现细节。
